# Find the underlying price for a target

MorphIQ SDK for Python

Find spot levels that make an option worth a target price or produce target
position P&L relative to the original model mark. Enter finite positive ordered
search bounds. Quantity is signed; multiplier is underlying units per contract.
Option-price targets are per underlying unit; P&L targets are position currency.
Volatility/rate/yield controls use percent; requests use decimals.

The approved flat rate is **100 CU per BSM request** or **200 CU per BS1993
request**, covering every internal pricing call. Quote the account's active
schedule and choose Max CU before executing. No-root or unresolved results and
post-commit numerical failures still carry the flat charge. This workspace does
no local root finding. Default tolerances are 1e-9 spot and 1e-6 target value;
edit the explicit request construction below for other positive tolerances.

Requires the scenario API deployment and matching account permissions. This SDK
candidate is held from merge/release until API deployment is verified.

Inputs start with synthetic values. Run all cells to display controls: setup
and changing controls send **zero API requests**. Quote sends one free utility
request. Execute sends one billable request, capped by Max CU, after a current
quote. Actions run sequentially. Quotes reserve nothing. Changes invalidate the
quote and displayed result; prior receipts remain in `history`.

A missing charge is **not reported**, never zero. No automatic retries are made.
Enter a key at the masked prompt or set `MORPHIQ_API_KEY`; set
`MORPHIQ_API_BASE_URL` to a controlled service when testing. Never save credentials
or outputs. The widget ceiling is limited to exact JavaScript integers; the
Python client accepts the full unsigned 64-bit range.


In [ ]:
import os
from getpass import getpass

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from morphiq_sdk import (
    APIError,
    MorphIQ,
    PriceRequest,
    ResponseError,
    ScenarioOptionPriceTarget,
    ScenarioPosition,
    ScenarioPositionPnlTarget,
    ScenarioSpotSolveRequest,
    TransportError,
)

api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")
model = widgets.Dropdown(
    options=[("BSM · European", "bsm"), ("BS1993 · American", "bs1993")], description="Model"
)
option_type = widgets.Dropdown(options=["call", "put"], description="Option")
spot = widgets.FloatText(value=100.0, description="Spot")
strike = widgets.FloatText(value=100.0, description="Strike")
expiry = widgets.FloatText(value=1.0, description="Years left")
volatility = widgets.FloatText(value=20.0, description="Volatility %")
rate = widgets.FloatText(value=5.0, description="Rate %")
yield_input = widgets.FloatText(value=0.0, description="Yield %")
multiplier = widgets.FloatText(value=100.0, description="Multiplier")
quote_button = widgets.Button(
    description="Quote · 1 request / 0 CU", layout=widgets.Layout(width="240px")
)
run_button = widgets.Button(
    description="Execute · 1 request", disabled=True, layout=widgets.Layout(width="240px")
)
output = widgets.Output()


def show_message(*values):
    output.append_stdout(" ".join(str(value) for value in values) + "\n")


usage = widgets.HTML(value="No API requests sent.")
history = []
last_quote = None
last_response = None
quote_snapshot = None
result_snapshot = None


def contract_inputs():
    return PriceRequest(
        exercise_style="american" if model.value == "bs1993" else "european",
        pricing_model="bjerksund_stensland_1993"
        if model.value == "bs1993"
        else "black_scholes_merton",
        option_type=option_type.value,
        spot=spot.value,
        strike=strike.value,
        time_to_expiry_years=expiry.value,
        volatility=volatility.value / 100,
        risk_free_rate=rate.value / 100,
        dividend_yield=yield_input.value / 100,
    )


quantity = widgets.FloatText(value=2.0, description="Quantity")
target_kind = widgets.Dropdown(
    options=[("Option price", "option_price"), ("Position P&L", "position_pnl")],
    description="Target kind",
)
target_value = widgets.FloatText(value=15.0, description="Target value")
min_spot = widgets.FloatText(value=50.0, description="Min spot")
max_spot = widgets.FloatText(value=150.0, description="Max spot")
max_cu = widgets.BoundedIntText(value=100, min=0, max=2**53 - 1, description="Max CU")
controls = [
    model,
    option_type,
    spot,
    strike,
    expiry,
    volatility,
    rate,
    yield_input,
    target_kind,
    target_value,
    quantity,
    multiplier,
    min_spot,
    max_spot,
]


def current_request():
    target = ScenarioOptionPriceTarget(kind="option_price", value=target_value.value)
    if target_kind.value == "position_pnl":
        target = ScenarioPositionPnlTarget(
            kind="position_pnl",
            value=target_value.value,
            position=ScenarioPosition(quantity=quantity.value, multiplier=multiplier.value),
        )
    return ScenarioSpotSolveRequest(
        contract=contract_inputs(), target=target, min_spot=min_spot.value, max_spot=max_spot.value
    )


def execute_request(client, request):
    return client.solve_spot(request, max_cu=max_cu.value)


def show_result(response):
    result = response.result
    show_message("Coverage:", result.coverage.status)
    show_message("Pricing evaluations (diagnostic, not a billing unit):", result.price_evaluations)
    if result.boundaries:
        display(
            pd.DataFrame(
                [boundary.model_dump(exclude_unset=True) for boundary in result.boundaries]
            )
        )
    else:
        show_message(
            "No boundaries found at the declared scan resolution. This is not a global proof."
        )
    output.append_display_data(pd.DataFrame([region.model_dump() for region in result.regions]))
    if result.coverage.status == "unresolved":
        show_message("Unresolved cells: inspect the bounds, best candidate, error and reason.")
        output.append_display_data(
            pd.DataFrame([cell.model_dump() for cell in result.coverage.cells])
        )
    show_message(
        "Point values and intervals are evaluated evidence. "
        "Resolved coverage can miss smaller structures."
    )
    show_message("Fixed assumptions for repricing a boundary:")
    output.append_display_data(pd.DataFrame([result.inputs.model_dump(exclude_unset=True)]))

In [ ]:
def record(action, request, charged, request_id):
    history.append(
        {"action": action, "request": request, "cu_charged": charged, "request_id": request_id}
    )
    known = sum(row["cu_charged"] for row in history if row["cu_charged"] is not None)
    unknown = sum(row["cu_charged"] is None for row in history)
    usage.value = (
        f"Recorded receipts: {known} CU; {unknown} unknown charges; {len(history)} requests."
    )


def changed(_change):
    global last_quote, last_response, quote_snapshot, result_snapshot
    last_quote = last_response = quote_snapshot = result_snapshot = None
    run_button.disabled = True
    with output:
        output.clear_output(wait=True)
        show_message(
            "Inputs changed. Quote again before execution. Previous receipts remain in history."
        )


for control in controls:
    control.observe(changed, names="value")


def perform(action):
    global last_quote, last_response, quote_snapshot, result_snapshot
    with output:
        output.clear_output(wait=True)
        try:
            request = current_request()
            wire = request.to_wire()
            if action == "execute":
                if last_quote is None or wire != quote_snapshot:
                    show_message("Quote the current inputs first. No execution submitted.")
                    return
                if last_quote.quote.total_fcu > max_cu.value:
                    show_message("Quote exceeds Max CU. No execution submitted.")
                    return
            else:
                last_quote = quote_snapshot = None
            last_response = result_snapshot = None
            for control in controls + [max_cu, quote_button, run_button]:
                control.disabled = True
            with MorphIQ(api_key=api_key) as client:
                if action == "quote":
                    response = client.quote(request)
                    last_quote, quote_snapshot = response, wire
                    record(
                        action, wire, response.metadata.fcu_charged, response.metadata.request_id
                    )
                    show_message(f"Quoted execution charge: {response.quote.total_fcu} CU")
                    show_message("Quote charge:", response.metadata.fcu_charged)
                    show_message(
                        "Quotes reserve nothing and do not lock policy or guarantee funds/capacity."
                    )
                else:
                    response = execute_request(client, request)
                    last_response, result_snapshot = response, wire
                    record(
                        action, wire, response.metadata.fcu_charged, response.metadata.request_id
                    )
                    last_quote = quote_snapshot = None
                    show_message(
                        "Commercial CU charged:",
                        response.metadata.fcu_charged
                        if response.metadata.fcu_charged is not None
                        else "not reported",
                    )
                    show_message("Request ID:", response.metadata.request_id)
                    show_result(response)
        except (APIError, ResponseError, TransportError) as error:
            metadata = getattr(error, "metadata", None)
            charged = 0 if action == "quote" else (metadata.fcu_charged if metadata else None)
            record(action, wire, charged, getattr(error, "request_id", None))
            last_quote = quote_snapshot = None
            show_message(str(error))
            show_message(
                "Commercial CU charged:", charged if charged is not None else "not reported"
            )
            show_message("No retry. A missing response is not evidence of zero charge.")
        except ValueError as error:
            show_message(f"Invalid local inputs/configuration: {error}. No request submitted.")
        finally:
            for control in controls + [max_cu, quote_button]:
                control.disabled = False
            run_button.disabled = last_quote is None


quote_button.on_click(lambda _button: perform("quote"))
run_button.on_click(lambda _button: perform("execute"))
display(widgets.VBox(controls + [max_cu, widgets.HBox([quote_button, run_button]), usage, output]))